<a href="https://colab.research.google.com/github/govardhankoduri46-png/Majorproject/blob/main/majorproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ADVANCED INSTANCE SEGMENTATION USING MASK R-CNN
# IMAGE-BASED GOOGLE COLAB PROJECT
# ============================================================

!pip install -q torch torchvision opencv-python pillow matplotlib pandas

import os
import csv
import json
import zipfile
import shutil
import random
import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import files
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn,
    MaskRCNN_ResNet50_FPN_Weights
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.70
MASK_THRESHOLD = 0.50
MIN_MASK_AREA = 100

OUTPUT_DIR = "advanced_segmentation_outputs"
CROPS_DIR = os.path.join(OUTPUT_DIR, "object_crops")
CUTOUTS_DIR = os.path.join(OUTPUT_DIR, "transparent_cutouts")

os.makedirs(CROPS_DIR, exist_ok=True)
os.makedirs(CUTOUTS_DIR, exist_ok=True)

# Reproducible colors
random.seed(42)

# ------------------------------------------------------------
# 2. Select Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 3. Load Pre-trained Mask R-CNN
# ------------------------------------------------------------

weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT

model = maskrcnn_resnet50_fpn(weights=weights)
model = model.to(device)
model.eval()

categories = weights.meta["categories"]
transform = weights.transforms()

print("Mask R-CNN model loaded successfully")

# ------------------------------------------------------------
# 4. Upload Input Image
# ------------------------------------------------------------

print("Upload your input image:")
uploaded = files.upload()

input_file = list(uploaded.keys())[0]
print("Input image:", input_file)

# ------------------------------------------------------------
# 5. Read Image
# ------------------------------------------------------------

image = Image.open(input_file).convert("RGB")
original = np.array(image)

height, width = original.shape[:2]

# Model preprocessing
image_tensor = transform(image).to(device)

# ------------------------------------------------------------
# 6. Prediction
# ------------------------------------------------------------

print("Running instance segmentation...")

with torch.no_grad():
    prediction = model([image_tensor])[0]

print("Prediction completed")

# ------------------------------------------------------------
# 7. Prepare Output Canvases
# ------------------------------------------------------------

annotated_image = original.copy()
colored_mask_image = np.zeros_like(original)
transparent_overlay = np.zeros(
    (height, width, 4),
    dtype=np.uint8
)

# Dashboard data
object_records = []
class_counts = {}

# Color generator
def generate_color(index):
    """
    Creates a visually distinct BGR color for each object.
    """
    hue = int((index * 137.5) % 180)
    hsv_color = np.uint8([[[hue, 220, 255]]])
    bgr_color = cv2.cvtColor(hsv_color, cv2.COLOR_HSV2BGR)[0][0]
    return tuple(int(value) for value in bgr_color)

# ------------------------------------------------------------
# 8. Process Detected Instances
# ------------------------------------------------------------

detected_objects = 0

for i in range(len(prediction["scores"])):

    score = float(prediction["scores"][i].item())

    if score < CONFIDENCE_THRESHOLD:
        continue

    mask_probability = (
        prediction["masks"][i, 0]
        .detach()
        .cpu()
        .numpy()
    )

    binary_mask = mask_probability >= MASK_THRESHOLD

    mask_area = int(np.sum(binary_mask))

    if mask_area < MIN_MASK_AREA:
        continue

    detected_objects += 1
    instance_id = detected_objects

    # Class label
    label_id = int(prediction["labels"][i].item())
    label = categories[label_id]

    # Bounding box
    box = (
        prediction["boxes"][i]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    x1, y1, x2, y2 = box

    # Keep coordinates within image boundaries
    x1 = max(0, min(x1, width - 1))
    y1 = max(0, min(y1, height - 1))
    x2 = max(0, min(x2, width - 1))
    y2 = max(0, min(y2, height - 1))

    box_width = max(0, x2 - x1)
    box_height = max(0, y2 - y1)

    # Object center
    mask_coordinates = np.where(binary_mask)

    if len(mask_coordinates[0]) > 0:
        center_y = int(np.mean(mask_coordinates[0]))
        center_x = int(np.mean(mask_coordinates[1]))
    else:
        center_x = int((x1 + x2) / 2)
        center_y = int((y1 + y2) / 2)

    # Unique object color in RGB and BGR
    color_bgr = generate_color(instance_id)
    color_rgb = (
        color_bgr[2],
        color_bgr[1],
        color_bgr[0]
    )

    # --------------------------------------------------------
    # 8.1 Create Colored Mask
    # --------------------------------------------------------

    colored_mask_image[binary_mask] = color_rgb

    # Transparent overlay
    transparent_overlay[binary_mask, 0] = color_rgb[0]
    transparent_overlay[binary_mask, 1] = color_rgb[1]
    transparent_overlay[binary_mask, 2] = color_rgb[2]
    transparent_overlay[binary_mask, 3] = 150

    # --------------------------------------------------------
    # 8.2 Draw Filled Segmentation Overlay
    # --------------------------------------------------------

    overlay = annotated_image.copy()
    overlay[binary_mask] = color_rgb

    annotated_image = cv2.addWeighted(
        annotated_image,
        0.65,
        overlay,
        0.35,
        0
    )

    # --------------------------------------------------------
    # 8.3 Draw Object Contour
    # --------------------------------------------------------

    mask_uint8 = (binary_mask.astype(np.uint8) * 255)

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    cv2.drawContours(
        annotated_image,
        contours,
        -1,
        color_rgb,
        3
    )

    # --------------------------------------------------------
    # 8.4 Draw Bounding Box
    # --------------------------------------------------------

    cv2.rectangle(
        annotated_image,
        (x1, y1),
        (x2, y2),
        color_rgb,
        2
    )

    # --------------------------------------------------------
    # 8.5 Draw Center Point
    # --------------------------------------------------------

    cv2.circle(
        annotated_image,
        (center_x, center_y),
        6,
        (255, 255, 255),
        -1
    )

    cv2.circle(
        annotated_image,
        (center_x, center_y),
        6,
        color_rgb,
        2
    )

    # --------------------------------------------------------
    # 8.6 Draw Label
    # --------------------------------------------------------

    label_text = f"ID {instance_id} | {label} | {score:.2f}"

    text_x = x1
    text_y = max(y1 - 10, 25)

    text_size, baseline = cv2.getTextSize(
        label_text,
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        2
    )

    text_width, text_height = text_size

    cv2.rectangle(
        annotated_image,
        (
            text_x,
            text_y - text_height - baseline - 5
        ),
        (
            text_x + text_width + 8,
            text_y + 5
        ),
        color_rgb,
        -1
    )

    cv2.putText(
        annotated_image,
        label_text,
        (text_x + 4, text_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    # --------------------------------------------------------
    # 8.7 Save Cropped Object Image
    # --------------------------------------------------------

    crop = original[y1:y2, x1:x2].copy()
    crop_mask = binary_mask[y1:y2, x1:x2]

    if crop.size > 0:
        crop_overlay = crop.copy()
        crop_overlay[crop_mask] = color_rgb

        crop_result = cv2.addWeighted(
            crop,
            0.65,
            crop_overlay,
            0.35,
            0
        )

        crop_filename = (
            f"object_{instance_id:03d}_{label}_"
            f"{score:.2f}.jpg"
        )

        crop_filename = crop_filename.replace("/", "_")
        crop_path = os.path.join(CROPS_DIR, crop_filename)

        cv2.imwrite(
            crop_path,
            cv2.cvtColor(crop_result, cv2.COLOR_RGB2BGR)
        )

        # ----------------------------------------------------
        # 8.8 Save Transparent Object Cutout
        # ----------------------------------------------------

        rgba_crop = np.zeros(
            (crop.shape[0], crop.shape[1], 4),
            dtype=np.uint8
        )

        rgba_crop[:, :, :3] = crop
        rgba_crop[:, :, 3] = (
            crop_mask.astype(np.uint8) * 255
        )

        cutout_filename = (
            f"object_{instance_id:03d}_{label}_transparent.png"
        )

        cutout_filename = cutout_filename.replace("/", "_")
        cutout_path = os.path.join(CUTOUTS_DIR, cutout_filename)

        Image.fromarray(rgba_crop).save(cutout_path)

    # --------------------------------------------------------
    # 8.9 Record Object Statistics
    # --------------------------------------------------------

    object_record = {
        "object_id": instance_id,
        "class_id": label_id,
        "class_name": label,
        "confidence": round(score, 4),
        "mask_area_pixels": mask_area,
        "mask_area_percent": round(
            (mask_area / (width * height)) * 100,
            4
        ),
        "bounding_box_x1": int(x1),
        "bounding_box_y1": int(y1),
        "bounding_box_x2": int(x2),
        "bounding_box_y2": int(y2),
        "bounding_box_width": int(box_width),
        "bounding_box_height": int(box_height),
        "center_x": int(center_x),
        "center_y": int(center_y),
        "color_rgb": str(color_rgb)
    }

    object_records.append(object_record)

    class_counts[label] = class_counts.get(label, 0) + 1

# ------------------------------------------------------------
# 9. Save Main Output Images
# ------------------------------------------------------------

annotated_path = os.path.join(
    OUTPUT_DIR,
    "01_annotated_instance_segmentation.jpg"
)

colored_mask_path = os.path.join(
    OUTPUT_DIR,
    "02_unique_instance_masks.png"
)

transparent_overlay_path = os.path.join(
    OUTPUT_DIR,
    "03_transparent_segmentation_overlay.png"
)

original_path = os.path.join(
    OUTPUT_DIR,
    "04_original_image.jpg"
)

cv2.imwrite(
    annotated_path,
    cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    colored_mask_path,
    cv2.cvtColor(colored_mask_image, cv2.COLOR_RGB2BGR)
)

Image.fromarray(transparent_overlay, mode="RGBA").save(
    transparent_overlay_path
)

cv2.imwrite(
    original_path,
    cv2.cvtColor(original, cv2.COLOR_RGB2BGR)
)

# ------------------------------------------------------------
# 10. Create Detection Report
# ------------------------------------------------------------

report_json_path = os.path.join(
    OUTPUT_DIR,
    "05_detection_report.json"
)

with open(report_json_path, "w") as json_file:
    json.dump(
        {
            "input_image": input_file,
            "image_width": width,
            "image_height": height,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "mask_threshold": MASK_THRESHOLD,
            "objects_detected": detected_objects,
            "objects": object_records
        },
        json_file,
        indent=4
    )

# ------------------------------------------------------------
# 11. Create CSV Report
# ------------------------------------------------------------

report_csv_path = os.path.join(
    OUTPUT_DIR,
    "06_object_statistics.csv"
)

if object_records:
    dataframe = pd.DataFrame(object_records)
    dataframe.to_csv(report_csv_path, index=False)
else:
    dataframe = pd.DataFrame()
    dataframe.to_csv(report_csv_path, index=False)

# ------------------------------------------------------------
# 12. Create Class Summary
# ------------------------------------------------------------

class_summary_path = os.path.join(
    OUTPUT_DIR,
    "07_class_summary.csv"
)

class_summary = pd.DataFrame(
    [
        {
            "class_name": class_name,
            "object_count": count
        }
        for class_name, count in sorted(class_counts.items())
    ]
)

class_summary.to_csv(class_summary_path, index=False)

# ------------------------------------------------------------
# 13. Generate Advanced Dashboard
# ------------------------------------------------------------

dashboard_path = os.path.join(
    OUTPUT_DIR,
    "08_analysis_dashboard.png"
)

fig = plt.figure(figsize=(20, 12))

# Original image
ax1 = plt.subplot(2, 3, 1)
ax1.imshow(original)
ax1.set_title("Original Image", fontsize=14, fontweight="bold")
ax1.axis("off")

# Annotated output
ax2 = plt.subplot(2, 3, 2)
ax2.imshow(annotated_image)
ax2.set_title(
    f"Detected Objects: {detected_objects}",
    fontsize=14,
    fontweight="bold"
)
ax2.axis("off")

# Unique instance masks
ax3 = plt.subplot(2, 3, 3)
ax3.imshow(colored_mask_image)
ax3.set_title("Unique Instance Masks", fontsize=14, fontweight="bold")
ax3.axis("off")

# Confidence chart
ax4 = plt.subplot(2, 3, 4)

if object_records:
    object_ids = [
        item["object_id"]
        for item in object_records
    ]

    confidence_values = [
        item["confidence"]
        for item in object_records
    ]

    bar_colors = [
        np.array(generate_color(object_id))[::-1] / 255.0
        for object_id in object_ids
    ]

    ax4.bar(
        object_ids,
        confidence_values,
        color=bar_colors
    )

    ax4.set_ylim(0, 1.05)
    ax4.set_xlabel("Object ID")
    ax4.set_ylabel("Confidence")
    ax4.set_title("Confidence by Object")
    ax4.grid(axis="y", alpha=0.3)

else:
    ax4.text(
        0.5,
        0.5,
        "No objects detected",
        ha="center",
        va="center"
    )
    ax4.axis("off")

# Class count chart
ax5 = plt.subplot(2, 3, 5)

if class_counts:
    class_names = list(class_counts.keys())
    class_values = list(class_counts.values())

    ax5.barh(
        class_names,
        class_values,
        color="#2E86DE"
    )

    ax5.set_xlabel("Number of Objects")
    ax5.set_title("Objects by Class")
    ax5.grid(axis="x", alpha=0.3)

else:
    ax5.text(
        0.5,
        0.5,
        "No classes detected",
        ha="center",
        va="center"
    )
    ax5.axis("off")

# Object area chart
ax6 = plt.subplot(2, 3, 6)

if object_records:
    area_values = [
        item["mask_area_percent"]
        for item in object_records
    ]

    labels = [
        f"ID {item['object_id']}"
        for item in object_records
    ]

    ax6.bar(
        labels,
        area_values,
        color="#10AC84"
    )

    ax6.set_xlabel("Object")
    ax6.set_ylabel("Image Area (%)")
    ax6.set_title("Object Mask Area")
    ax6.tick_params(axis="x", rotation=45)
    ax6.grid(axis="y", alpha=0.3)

else:
    ax6.text(
        0.5,
        0.5,
        "No area data",
        ha="center",
        va="center"
    )
    ax6.axis("off")

plt.suptitle(
    "Advanced Mask R-CNN Instance Segmentation Dashboard",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout()
plt.savefig(
    dashboard_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 14. Display Main Results
# ------------------------------------------------------------

plt.figure(figsize=(20, 8))

plt.subplot(1, 3, 1)
plt.imshow(original)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(annotated_image)
plt.title("Advanced Segmentation")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(colored_mask_image)
plt.title("Instance ID Masks")
plt.axis("off")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 15. Print Results
# ------------------------------------------------------------

print("=" * 60)
print("ADVANCED SEGMENTATION COMPLETED")
print("=" * 60)
print("Objects detected:", detected_objects)
print("Image size:", f"{width} x {height}")
print("Classes detected:", class_counts)
print("Output directory:", OUTPUT_DIR)

if object_records:
    print("\nObject statistics:")
    display(dataframe)

# ------------------------------------------------------------
# 16. Create ZIP Package
# ------------------------------------------------------------

zip_filename = "advanced_mask_rcnn_segmentation_results"

shutil.make_archive(
    zip_filename,
    "zip",
    OUTPUT_DIR
)

zip_path = zip_filename + ".zip"

print("\nZIP package created:", zip_path)

# ------------------------------------------------------------
# 17. Download Only the ZIP File
# ------------------------------------------------------------

files.download(zip_path)

Using device: cpu
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:01<00:00, 138MB/s]


Mask R-CNN model loaded successfully
Upload your input image:
